# Bronze Layer: CSV Ingestion to Delta Tables

Reads raw CSV files from the Unity Catalog volume `/Volumes/automotive_project/bronze/automotive_raw/`, adds an `ingestion_timestamp` column, and writes each file as a Delta table in `automotive_project.bronze`. The CSV filename (without extension) becomes the Delta table name. Original CSV files are never modified.

In [0]:
# ─── Configuration ───────────────────────────────────────────────────────────
# Update these variables to reuse this notebook for a different catalog,
# schema, or volume path.
CATALOG = "automotive_project"
SCHEMA = "bronze"
VOLUME_PATH = "/Volumes/automotive_project/bronze/automotive_raw"

FULL_SCHEMA = f"{CATALOG}.{SCHEMA}"

In [0]:
import re
from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
)


# ─── Helper functions ─────────────────────────────────────────────────────────
def sanitize_table_name(filename: str) -> str:
    """Convert a CSV filename to a valid, lowercase Delta table name."""
    name = filename.removesuffix(".csv")
    name = re.sub(r"[^a-zA-Z0-9_]", "_", name)
    if name and name[0].isdigit():
        name = f"t_{name}"
    return name.lower()


def ingest_csv_to_delta(csv_name: str, volume_path: str, catalog_schema: str) -> dict:
    """Read a CSV from a UC Volume and write it as a Delta table.

    Parameters
    ----------
    csv_name : str
        Filename including the ``.csv`` extension (e.g. ``customers.csv``).
    volume_path : str
        UC Volume directory path (e.g. ``/Volumes/cat/sch/vol``).
    catalog_schema : str
        Fully-qualified schema, e.g. ``automotive_project.bronze``.

    Returns
    -------
    dict
        Result record with keys: ``csv_file``, ``table_name``, ``row_count``,
        ``column_count``, ``status``, ``error_message``.
    """
    file_path = f"{volume_path.rstrip('/')}/{csv_name}"
    table_name = sanitize_table_name(csv_name)
    full_table_name = f"{catalog_schema}.{table_name}"

    try:
        df = (
            spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .csv(file_path)
        )

        # Add ingestion timestamp (read-only — original CSV files are untouched)
        df = df.withColumn("ingestion_timestamp", current_timestamp())

        (
            df.write
            .mode("overwrite")
            .format("delta")
            .saveAsTable(full_table_name)
        )

        row_count = df.count()
        col_count = len(df.columns)

        return {
            "csv_file": csv_name,
            "table_name": full_table_name,
            "row_count": row_count,
            "column_count": col_count,
            "status": "SUCCESS",
            "error_message": "",
        }
    except Exception as exc:
        return {
            "csv_file": csv_name,
            "table_name": full_table_name,
            "row_count": 0,
            "column_count": 0,
            "status": "FAILED",
            "error_message": str(exc),
        }


# ─── Ensure target schema exists ─────────────────────────────────────────────
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {FULL_SCHEMA}")

# ─── List CSV files in the volume ────────────────────────────────────────────
all_entries = dbutils.fs.ls(VOLUME_PATH)
csv_files = [
    e.name
    for e in all_entries
    if e.name.endswith(".csv") and not e.isDir()
]

print(f"Found {len(csv_files)} CSV file(s) in {VOLUME_PATH}\n")
for fname in sorted(csv_files):
    print(f"  • {fname}")

# ─── Ingest each CSV file ────────────────────────────────────────────────────
results = []
for fname in sorted(csv_files):
    result = ingest_csv_to_delta(fname, VOLUME_PATH, FULL_SCHEMA)
    results.append(result)
    if result["status"] == "SUCCESS":
        print(
            f"  ✓ {result['table_name']}  "
            f"({result['row_count']} rows, {result['column_count']} columns)"
        )
    else:
        print(f"  ✗ {result['table_name']}  — {result['error_message']}")

success_count = sum(1 for r in results if r["status"] == "SUCCESS")
fail_count = sum(1 for r in results if r["status"] == "FAILED")
print(f"\nIngestion complete: {success_count} succeeded, {fail_count} failed.")

Found 15 CSV file(s) in /Volumes/automotive_project/bronze/automotive_raw

  • addresses.csv
  • customer_vehicles.csv
  • customers.csv
  • dealers.csv
  • parts_catalog.csv
  • parts_used.csv
  • service_appointments.csv
  • service_centers.csv
  • service_order_tasks.csv
  • service_orders.csv
  • service_types.csv
  • technicians.csv
  • vehicle_models.csv
  • vehicles.csv
  • warranty_claims.csv
  ✓ automotive_project.bronze.addresses  (10100 rows, 7 columns)
  ✓ automotive_project.bronze.customer_vehicles  (10000 rows, 8 columns)
  ✓ automotive_project.bronze.customers  (10000 rows, 8 columns)
  ✓ automotive_project.bronze.dealers  (20 rows, 6 columns)
  ✓ automotive_project.bronze.parts_catalog  (50 rows, 6 columns)
  ✓ automotive_project.bronze.parts_used  (500000 rows, 7 columns)
  ✓ automotive_project.bronze.service_appointments  (10000 rows, 7 columns)
  ✓ automotive_project.bronze.service_centers  (50 rows, 6 columns)
  ✓ automotive_project.bronze.service_order_tasks  (5000

In [0]:
# ─── Summary of Bronze tables created ────────────────────────────────────────
if results:
    summary_schema = StructType(
        [
            StructField("csv_file", StringType(), True),
            StructField("table_name", StringType(), True),
            StructField("row_count", LongType(), True),
            StructField("column_count", LongType(), True),
            StructField("status", StringType(), True),
            StructField("error_message", StringType(), True),
        ]
    )

    print("Ingestion Summary")
    display(spark.createDataFrame(results, schema=summary_schema))

    print(f"\nAll tables in {FULL_SCHEMA}:")
    display(spark.sql(f"SHOW TABLES IN {FULL_SCHEMA}"))
else:
    print("No CSV files found — no tables were created.")

Ingestion Summary


csv_file,table_name,row_count,column_count,status,error_message
addresses.csv,automotive_project.bronze.addresses,10100,7,SUCCESS,
customer_vehicles.csv,automotive_project.bronze.customer_vehicles,10000,8,SUCCESS,
customers.csv,automotive_project.bronze.customers,10000,8,SUCCESS,
dealers.csv,automotive_project.bronze.dealers,20,6,SUCCESS,
parts_catalog.csv,automotive_project.bronze.parts_catalog,50,6,SUCCESS,
parts_used.csv,automotive_project.bronze.parts_used,500000,7,SUCCESS,
service_appointments.csv,automotive_project.bronze.service_appointments,10000,7,SUCCESS,
service_centers.csv,automotive_project.bronze.service_centers,50,6,SUCCESS,
service_order_tasks.csv,automotive_project.bronze.service_order_tasks,500000,6,SUCCESS,
service_orders.csv,automotive_project.bronze.service_orders,10000,8,SUCCESS,



All tables in automotive_project.bronze:


database,tableName,isTemporary
bronze,addresses,false
bronze,customer_vehicles,false
bronze,customers,false
bronze,dealers,false
bronze,parts_catalog,false
bronze,parts_used,false
bronze,service_appointments,false
bronze,service_centers,false
bronze,service_order_tasks,false
bronze,service_orders,false


In [0]:
tables = [
    "addresses",
    "customer_vehicles",
    "customers",
    "dealers",
    "parts_catalog",
    "parts_used",
    "service_appointments",
    "service_centers",
    "service_order_tasks",
    "service_orders",
    "service_types",
    "technicians",
    "vehicle_models",
    "vehicles",
    "warranty_claims"
]

for table in tables:
    print(f"\n{'='*60}")
    print(f"TABLE: {table}")
    print(f"{'='*60}")
    spark.sql(f"DESCRIBE automotive_project.bronze.{table}").show(truncate=False)


TABLE: addresses
+-------------------+---------+-------+
|col_name           |data_type|comment|
+-------------------+---------+-------+
|address_id         |int      |NULL   |
|street_address     |string   |NULL   |
|city               |string   |NULL   |
|state              |string   |NULL   |
|postal_code        |int      |NULL   |
|country            |string   |NULL   |
|ingestion_timestamp|timestamp|NULL   |
+-------------------+---------+-------+


TABLE: customer_vehicles
+-------------------+---------+-------+
|col_name           |data_type|comment|
+-------------------+---------+-------+
|ownership_id       |int      |NULL   |
|customer_id        |int      |NULL   |
|vehicle_id         |int      |NULL   |
|registration_plate |string   |NULL   |
|purchase_dealer_id |int      |NULL   |
|purchase_date      |date     |NULL   |
|is_active          |boolean  |NULL   |
|ingestion_timestamp|timestamp|NULL   |
+-------------------+---------+-------+


TABLE: customers
+---------------